# 06 — Reason Codes
Turn model predictions into human-readable *why*, not just a label.

In [1]:
import sys
sys.path.insert(0, "..")
import pandas as pd, joblib
from _lib import generate_reason_code, recommend_action

val = pd.read_parquet("data_cache/val.parquet")
model = joblib.load("data_cache/model.joblib")
feature_cols = joblib.load("data_cache/feature_cols.joblib")

val["predicted_label"] = model.predict(val[feature_cols])
val["reason_code"] = val.apply(generate_reason_code, axis=1)
val["action"] = val.apply(lambda r: recommend_action(r["predicted_label"], r), axis=1)

val[["page_id","predicted_label","reason_code","action"]].head(10)

,page_id,predicted_label,reason_code,action
0,page_0000,stable,"content is 2+ years old, candidate for refresh",monitor
2,page_0002,stable,clicks trending down within observation window,monitor
9,page_0009,stable,CTR below typical range for its position; cont...,monitor
11,page_0011,declining,clicks trending down within observation window,improve
15,page_0015,stable,clicks trending down within observation window,monitor
18,page_0018,stable,"content is 2+ years old, candidate for refresh",monitor
19,page_0019,stable,clicks trending up within observation window,monitor
22,page_0022,stable,"content is 2+ years old, candidate for refresh",monitor
25,page_0025,stable,"content is 2+ years old, candidate for refresh",improve
30,page_0030,growing,clicks trending up within observation window; ...,protect


In [2]:
val.to_parquet("data_cache/val_with_reasons.parquet")
print("Saved reason-coded validation set.")

Saved reason-coded validation set.


## Notes
Reason codes are template-based off feature deltas (trend slope, CTR vs typical
range, position, content age) rather than free-text generation — keeps them
auditable and directly traceable to a specific number, which matters for a
content team trusting the recommendation.